In [ ]:
import yfinance as yf
from io import BytesIO
import requests
import pdfplumber
import datetime
from datetime import timedelta
import re

# Get close price of Brent crude oil
def get_ticker(ticker):
    brent = yf.Ticker(ticker)
    data = brent.history(period="1d")
    return data['Close'].iloc[0], data.index[0]


def request_market_data(code):
    close_price, close_day = get_ticker(code)
    close_day_utc = close_day.tz_convert("UTC")
    return round(close_price,2), close_day_utc


request_time = datetime.datetime.now(tz=datetime.timezone.utc)
metrics = ["BZ=F", "USDAUD=X", "DX-Y.NYB"]

market_data = []
for metric in metrics:
    price, close_timestamp = request_market_data(metric)
    market_data.append({"metric" :metric, "price": price, "close_timestamp": close_timestamp, "request_time": request_time})

In [ ]:
def find_last_sunday():
    today = datetime.datetime.now().weekday()
    days = (today+1) % 7
    if days == 0:
        days = 7
    return days

def get_latest_aip_report_url(daydelay: int) -> str:
    """Build URL for the most recent Sunday AIP report."""
    # AIP publishes on Sundays — find the last Sunday
    today = datetime.datetime.now() - timedelta(days=daydelay)  # Ensure we get last Sunday's report even if today is Sunday
    #print("Waring: Dev lagged by a week")
    days_since_sunday = today.weekday() + 1  # Monday=0, so Sunday=-1 mod 7
    last_sunday = today - timedelta(days=days_since_sunday % 7)
    
    month_str = last_sunday.strftime("%Y-%m")
    date_str = last_sunday.strftime("%d %B %Y")  # e.g. "5 April 2026"
    
    return (
        f"https://www.aip.com.au/sites/default/files/download-files/"
        f"{month_str}/Weekly%20Petrol%20Prices%20Report%20-%20{date_str.replace(' ', '%20')}.pdf"
    )  
    
def extract_mogas_95_price_from_pdf(response) -> str:
    with pdfplumber.open(BytesIO(response.content)) as pdf:
        first_page = pdf.pages[3]
        tables = first_page.extract_tables()
        
        if not tables:
            print("No tables found on first page")
            return None
        
        # Inspect the first table
        first_table = tables[0]
        mogas_95 = first_table[0][1].split("Average")[0].strip()
        return first_table[0][0], mogas_95
    
    
def extract_mogas_95():
    daydelay = find_last_sunday()
    report_url = get_latest_aip_report_url(daydelay)

    response = requests.get(report_url, timeout=15)
        
    if response.status_code != 200:
        print(f"Could not fetch AIP report: {report_url}")
        #return None
        
    mogas_95_label, mogas_95_price = extract_mogas_95_price_from_pdf(response)
    match = re.search(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b", mogas_95_label)
    mogas_95_date = match.group(0) if match else None
    return mogas_95_price, mogas_95_date


mogas_95_price, mogas_95_date = extract_mogas_95()
market_data.append({"metric" : "Mogas 95", "price": mogas_95_price, "close_timestamp": mogas_95_date, "request_time": request_time})

In [89]:
market_data

[{'metric': 'BZ=F',
  'price': np.float64(108.17),
  'close_timestamp': Timestamp('2026-05-01 04:00:00+0000', tz='UTC'),
  'request_time': datetime.datetime(2026, 5, 3, 1, 34, 14, 749746, tzinfo=datetime.timezone.utc)},
 {'metric': 'USDAUD=X',
  'price': np.float64(1.39),
  'close_timestamp': Timestamp('2026-04-30 23:00:00+0000', tz='UTC'),
  'request_time': datetime.datetime(2026, 5, 3, 1, 34, 14, 749746, tzinfo=datetime.timezone.utc)},
 {'metric': 'DX-Y.NYB',
  'price': np.float64(98.21),
  'close_timestamp': Timestamp('2026-05-01 04:00:00+0000', tz='UTC'),
  'request_time': datetime.datetime(2026, 5, 3, 1, 34, 14, 749746, tzinfo=datetime.timezone.utc)},
 {'metric': 'Mogas 95',
  'price': '111.8',
  'close_timestamp': '24/04/26',
  'request_time': datetime.datetime(2026, 5, 3, 1, 34, 14, 749746, tzinfo=datetime.timezone.utc)}]